# Data object modeling

Everything amoorion takes and gives is a model (pydantic): checked when made, with a
schema, and unknown fields are refused.

In [1]:
# The shared models: what every platform reads and writes.
from amoorion.models import Account, Message, Post, Result

# A post is the same for all platforms; each plugin makes its own form of it.
post = Post(
    title="Hello",
    text="the text",
    tags=["news"],
    links=["https://example.org"],
)

# The last line of a cell gets displayed.
post

Post(text='the text', title='Hello', links=['https://example.org'], tags=['news'])

In [2]:
# Models check themselves: an empty post can't exist…
try:
    Post(text="")

except ValueError as error:
    print(error.errors()[0]["msg"])


# …and neither can a field nobody knows, so typos don't get lost silently.
try:
    Post(
        text="x",
        colour="red",
    )

except ValueError as error:
    print(error.errors()[0]["msg"])

String should have at least 1 character
Extra inputs are not permitted


A message is something said on a platform; JSON there and back:

In [3]:
# A message as a platform gave it, with its author.
message = Message(
    platform="matrix",
    text="hi",
    id="1",
    author=Account(
        platform="matrix",
        handle="ada",
    ),
)

# To JSON and back: this is how it travels and how it is stored.
restored = Message.model_validate_json(message.model_dump_json())

# Nothing was lost on the way.
restored == message

True

In [4]:
# Models also describe themselves, which is where the API and MCP get their schemas.
Post.model_json_schema()["properties"].keys()

dict_keys(['text', 'title', 'links', 'tags'])

A model of your own is the same thing, and views present any of them:

In [5]:
# The base of it all, for models of your own.
from amoorion.core import JsonView, Model


# Your own model: only the fields, the rest comes from `Model`.
class Release(Model):
    name: str
    stable: bool = False


# A view decides the output; JsonView gives plain data.
JsonView().render(
    [
        Release(name="v1"),
        Release(
            name="v2",
            stable=True,
        ),
    ]
)

[{'name': 'v1', 'stable': False}, {'name': 'v2', 'stable': True}]